# 📦 BhashaLens — Notebook 1: Data Collection & Cleaning

This notebook downloads, cleans, mixes, and splits translation datasets for MarianMT training.

**Language Pairs:** Hindi, Marathi, Tamil, Gujarati ↔ English (8 bidirectional models)

**Sources:** IIT Bombay, Samanantar, IndicTrans2, IndicCorp, PMIndia, OPUS, Custom Domain

**Runs locally** — no Kaggle or Colab required.

---

## 1. Setup & Dependencies

In [1]:
%pip install -q datasets opustools langdetect sentencepiece pyyaml pandas tqdm python-dotenv

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
import os
import re
import json
import hashlib
import unicodedata
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import yaml
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42
np.random.seed(42)

print('All dependencies loaded.')

d:\BHASHALENS1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All dependencies loaded.


In [3]:
# ── Local Paths ──
BASE_DIR = Path('./bhashalens_ml')

DATA_RAW = BASE_DIR / 'data' / 'raw'
DATA_MIXED = BASE_DIR / 'data' / 'mixed'
DATA_CLEANED = BASE_DIR / 'data' / 'cleaned'
REPORTS_DIR = BASE_DIR / 'reports'

for d in [DATA_RAW, DATA_MIXED, DATA_CLEANED, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Base directory: {BASE_DIR}')

Base directory: bhashalens_ml


In [4]:
# ── Configuration ──
CONFIG = {
    'language_pairs': ['hi-en', 'mr-en', 'ta-en', 'gu-en'],
    'lang_codes': {
        'hi': 'Hindi', 'en': 'English', 'mr': 'Marathi',
        'ta': 'Tamil', 'gu': 'Gujarati'
    },
    'mixing_ratios': {
        'hi-en': {
            'iit_bombay': 0.20, 'samanantar': 0.25, 'indictrans2': 0.15,
            'indiccorp': 0.10, 'pmindia': 0.10, 'opus': 0.10, 'custom': 0.10,
        },
        'mr-en': {
            'samanantar': 0.25, 'indictrans2': 0.20, 'indiccorp': 0.15,
            'pmindia': 0.15, 'opus': 0.15, 'custom': 0.10,
        },
        'ta-en': {
            'samanantar': 0.25, 'indictrans2': 0.20, 'indiccorp': 0.15,
            'pmindia': 0.15, 'opus': 0.15, 'custom': 0.10,
        },
        'gu-en': {
            'samanantar': 0.25, 'indictrans2': 0.20, 'indiccorp': 0.15,
            'pmindia': 0.15, 'opus': 0.15, 'custom': 0.10,
        },
    },
    'split_ratios': (0.90, 0.05, 0.05),
    'cleaning': {
        'max_length': 512,
        'max_length_ratio': 3.0,
        'lang_detect_confidence': 0.9,
    }
}

# The "reverse" pairs (en-hi, en-mr, etc.) use the same data with src/tgt swapped.
# We only need to collect data for the 4 base pairs.
print('Configuration loaded.')
print(f'Language pairs to collect: {CONFIG["language_pairs"]}')

Configuration loaded.
Language pairs to collect: ['hi-en', 'mr-en', 'ta-en', 'gu-en']


## 2. Dataset Download

### 2.1 IIT Bombay Hindi-English Corpus

In [5]:
def download_iit_bombay() -> pd.DataFrame:
    """Download IIT Bombay Hindi-English parallel corpus from HuggingFace."""
    print('Downloading IIT Bombay Hindi-English corpus...')
    ds = load_dataset('cfilt/iitb-english-hindi', split='train')
    
    pairs = []
    for item in tqdm(ds, desc='Processing IIT Bombay'):
        translation = item.get('translation', item)
        if isinstance(translation, dict):
            src = translation.get('hi', '')
            tgt = translation.get('en', '')
        else:
            continue
        pairs.append({'source': src, 'target': tgt})
    
    df = pd.DataFrame(pairs)
    save_path = DATA_RAW / 'hi-en' / 'iit_bombay'
    save_path.mkdir(parents=True, exist_ok=True)
    df.to_csv(save_path / 'data.tsv', sep='\t', index=False)
    
    print(f'IIT Bombay: {len(df):,} pairs saved')
    return df

iit_bombay_df = download_iit_bombay()

Processing IIT Bombay: 100%|██████████| 1659083/1659083 [00:25<00:00, 65862.84it/s]


IIT Bombay: 1,659,083 pairs saved


### 2.2 AI4Bharat Samanantar

In [6]:
def download_samanantar(lang_pair: str) -> pd.DataFrame:
    """Download AI4Bharat Samanantar dataset for a language pair."""
    src_lang = lang_pair.split('-')[0]
    print(f'Downloading Samanantar for {lang_pair}...')
    
    try:
        ds = load_dataset('ai4bharat/samanantar', src_lang, split='train')
    except Exception as e:
        print(f'Primary download failed: {e}')
        print('Trying alternative config...')
        try:
            ds = load_dataset('ai4bharat/samanantar', f'{src_lang}-en', split='train')
        except Exception as e2:
            print(f'Alternative also failed: {e2}')
            print(f'>> You may need to manually download Samanantar for {lang_pair}')
            print(f'>> Visit: https://ai4bharat.iitm.ac.in/samanantar/')
            return pd.DataFrame(columns=['source', 'target'])
    
    pairs = []
    for item in tqdm(ds, desc=f'Processing Samanantar {lang_pair}'):
        src = item.get('src', item.get(src_lang, ''))
        tgt = item.get('tgt', item.get('en', ''))
        if not src and not tgt:
            translation = item.get('translation', {})
            src = translation.get(src_lang, '')
            tgt = translation.get('en', '')
        pairs.append({'source': src, 'target': tgt})
    
    df = pd.DataFrame(pairs)
    save_path = DATA_RAW / lang_pair / 'samanantar'
    save_path.mkdir(parents=True, exist_ok=True)
    df.to_csv(save_path / 'data.tsv', sep='\t', index=False)
    
    print(f'Samanantar {lang_pair}: {len(df):,} pairs saved')
    return df

samanantar_data = {}
for lp in CONFIG['language_pairs']:
    samanantar_data[lp] = download_samanantar(lp)

Processing Samanantar hi-en: 100%|██████████| 10125706/10125706 [03:21<00:00, 50325.20it/s]


Samanantar hi-en: 10,125,706 pairs saved


Processing Samanantar mr-en: 100%|██████████| 3627480/3627480 [01:07<00:00, 53765.63it/s]


Samanantar mr-en: 3,627,480 pairs saved


Processing Samanantar ta-en: 100%|██████████| 5264867/5264867 [01:46<00:00, 49600.95it/s]


Samanantar ta-en: 5,264,867 pairs saved


Processing Samanantar gu-en: 100%|██████████| 3067790/3067790 [00:56<00:00, 54627.94it/s]


Samanantar gu-en: 3,067,790 pairs saved


### 2.3 IndicTrans2 Datasets (IN22-Gen & IN22-Conv)

In [7]:
def download_indictrans2(lang_pair: str) -> pd.DataFrame:
    """Download IndicTrans2 parallel data (IN22-Gen and IN22-Conv)."""
    hf_token = os.environ.get('HUGGINGFACE_TOKEN') or os.environ.get('HF_TOKEN')
    if not hf_token:
        print('WARNING: No HUGGINGFACE_TOKEN found in environment or .env file')

    src_lang = lang_pair.split('-')[0]
    print(f'Downloading IndicTrans2 for {lang_pair}...')
    
    all_pairs = []
    
    # Map language codes to IndicTrans2 naming conventions
    lang_map = {'hi': 'hin_Deva', 'mr': 'mar_Deva', 'ta': 'tam_Taml', 'gu': 'guj_Gujr', 'en': 'eng_Latn'}
    src_code = lang_map.get(src_lang, src_lang)
    
    for dataset_name in ['ai4bharat/IN22-Gen', 'ai4bharat/IN22-Conv']:
        short_name = dataset_name.split('/')[-1]
        print(f'  Trying {short_name}...')
        try:
            ds = load_dataset(dataset_name, split='test', token=hf_token)
            for item in tqdm(ds, desc=f'{short_name} {lang_pair}'):
                # Try different field name patterns
                src = item.get(src_code, item.get(src_lang, ''))
                tgt = item.get('eng_Latn', item.get('en', ''))
                if not src or not tgt:
                    sentence = item.get('sentence', {})
                    if isinstance(sentence, dict):
                        src = sentence.get(src_code, sentence.get(src_lang, ''))
                        tgt = sentence.get('eng_Latn', sentence.get('en', ''))
                if src and tgt:
                    all_pairs.append({'source': str(src), 'target': str(tgt)})
            print(f'  {short_name}: {len(all_pairs):,} pairs so far')
        except Exception as e:
            print(f'  {short_name} failed: {e}')
    
    df = pd.DataFrame(all_pairs) if all_pairs else pd.DataFrame(columns=['source', 'target'])
    save_path = DATA_RAW / lang_pair / 'indictrans2'
    save_path.mkdir(parents=True, exist_ok=True)
    df.to_csv(save_path / 'data.tsv', sep='\t', index=False)
    
    print(f'IndicTrans2 {lang_pair}: {len(df):,} pairs saved')
    return df

indictrans2_data = {}
for lp in CONFIG['language_pairs']:
    indictrans2_data[lp] = download_indictrans2(lp)

  Trying IN22-Gen...
  IN22-Gen failed: Dataset 'ai4bharat/IN22-Gen' is a gated dataset on the Hub. You must be authenticated to access it.
  Trying IN22-Conv...
  IN22-Conv failed: Dataset 'ai4bharat/IN22-Conv' is a gated dataset on the Hub. You must be authenticated to access it.
IndicTrans2 hi-en: 0 pairs saved
  Trying IN22-Gen...
  IN22-Gen failed: Dataset 'ai4bharat/IN22-Gen' is a gated dataset on the Hub. You must be authenticated to access it.
  Trying IN22-Conv...
  IN22-Conv failed: Dataset 'ai4bharat/IN22-Conv' is a gated dataset on the Hub. You must be authenticated to access it.
IndicTrans2 mr-en: 0 pairs saved
  Trying IN22-Gen...


  IN22-Gen failed: Dataset 'ai4bharat/IN22-Gen' is a gated dataset on the Hub. You must be authenticated to access it.
  Trying IN22-Conv...
  IN22-Conv failed: Dataset 'ai4bharat/IN22-Conv' is a gated dataset on the Hub. You must be authenticated to access it.
IndicTrans2 ta-en: 0 pairs saved
  Trying IN22-Gen...
  IN22-Gen failed: Dataset 'ai4bharat/IN22-Gen' is a gated dataset on the Hub. You must be authenticated to access it.
  Trying IN22-Conv...
  IN22-Conv failed: Dataset 'ai4bharat/IN22-Conv' is a gated dataset on the Hub. You must be authenticated to access it.
IndicTrans2 gu-en: 0 pairs saved


### 2.4 AI4Bharat IndicCorp

In [8]:
def download_indiccorp(lang_pair: str, max_sentences: int = 100000) -> pd.DataFrame:
    """Download AI4Bharat IndicCorp monolingual data.
    
    IndicCorp is monolingual, so we extract sentences for potential
    back-translation or data augmentation. We pair Indic sentences
    with empty targets (to be filtered or used for augmentation later).
    
    NOTE: If parallel data is available, it will be extracted as pairs.
    Otherwise this creates a monolingual corpus for augmentation.
    """
    src_lang = lang_pair.split('-')[0]
    print(f'Downloading IndicCorp for {lang_pair}...')
    
    pairs = []
    try:
        ds = load_dataset('ai4bharat/IndicCorp', src_lang, split='train',
                         streaming=True)
        count = 0
        for item in tqdm(ds, desc=f'IndicCorp {lang_pair}', total=max_sentences):
            text = item.get('text', '')
            if text and len(text.strip()) > 10:
                pairs.append({'source': text.strip(), 'target': ''})
                count += 1
                if count >= max_sentences:
                    break
        print(f'  IndicCorp: collected {len(pairs):,} monolingual sentences')
    except Exception as e:
        print(f'  IndicCorp download failed: {e}')
        print(f'  >> Visit: https://ai4bharat.iitm.ac.in/corpora')
    
    df = pd.DataFrame(pairs) if pairs else pd.DataFrame(columns=['source', 'target'])
    save_path = DATA_RAW / lang_pair / 'indiccorp'
    save_path.mkdir(parents=True, exist_ok=True)
    df.to_csv(save_path / 'data.tsv', sep='\t', index=False)
    
    print(f'IndicCorp {lang_pair}: {len(df):,} sentences saved')
    return df

indiccorp_data = {}
for lp in CONFIG['language_pairs']:
    indiccorp_data[lp] = download_indiccorp(lp)

  IndicCorp download failed: Dataset 'ai4bharat/IndicCorp' doesn't exist on the Hub or cannot be accessed.
  >> Visit: https://ai4bharat.iitm.ac.in/corpora
IndicCorp hi-en: 0 sentences saved
  IndicCorp download failed: Dataset 'ai4bharat/IndicCorp' doesn't exist on the Hub or cannot be accessed.
  >> Visit: https://ai4bharat.iitm.ac.in/corpora
IndicCorp mr-en: 0 sentences saved
  IndicCorp download failed: Dataset 'ai4bharat/IndicCorp' doesn't exist on the Hub or cannot be accessed.
  >> Visit: https://ai4bharat.iitm.ac.in/corpora
IndicCorp ta-en: 0 sentences saved
  IndicCorp download failed: Dataset 'ai4bharat/IndicCorp' doesn't exist on the Hub or cannot be accessed.
  >> Visit: https://ai4bharat.iitm.ac.in/corpora
IndicCorp gu-en: 0 sentences saved


### 2.5 PMIndia Parallel Corpus

In [9]:
def download_pmindia(lang_pair: str) -> pd.DataFrame:
    """Download PMIndia parallel corpus (PM speeches translated to Indian languages)."""
    src_lang = lang_pair.split('-')[0]
    print(f'Downloading PMIndia for {lang_pair}...')
    
    pairs = []
    try:
        # PMIndia uses language pair configs like "en-hi", "en-mr", etc.
        config_name = f'{src_lang}-en'
        try:
            ds = load_dataset('pmindia', config_name, split='train')
        except Exception:
            config_name = f'en-{src_lang}'
            ds = load_dataset('pmindia', config_name, split='train')
        
        for item in tqdm(ds, desc=f'PMIndia {lang_pair}'):
            translation = item.get('translation', item)
            if isinstance(translation, dict):
                src = translation.get(src_lang, '')
                tgt = translation.get('en', '')
            else:
                src = item.get(src_lang, '')
                tgt = item.get('en', '')
            if src and tgt:
                pairs.append({'source': str(src), 'target': str(tgt)})
        
        print(f'  PMIndia: {len(pairs):,} pairs downloaded')
    except Exception as e:
        print(f'  PMIndia download failed: {e}')
        print(f'  >> Visit: https://data.statmt.org/pmindia/')
    
    df = pd.DataFrame(pairs) if pairs else pd.DataFrame(columns=['source', 'target'])
    save_path = DATA_RAW / lang_pair / 'pmindia'
    save_path.mkdir(parents=True, exist_ok=True)
    df.to_csv(save_path / 'data.tsv', sep='\t', index=False)
    
    print(f'PMIndia {lang_pair}: {len(df):,} pairs saved')
    return df

pmindia_data = {}
for lp in CONFIG['language_pairs']:
    pmindia_data[lp] = download_pmindia(lp)

  PMIndia download failed: Dataset 'pmindia' doesn't exist on the Hub or cannot be accessed.
  >> Visit: https://data.statmt.org/pmindia/
PMIndia hi-en: 0 pairs saved
  PMIndia download failed: Dataset 'pmindia' doesn't exist on the Hub or cannot be accessed.
  >> Visit: https://data.statmt.org/pmindia/
PMIndia mr-en: 0 pairs saved
  PMIndia download failed: Dataset 'pmindia' doesn't exist on the Hub or cannot be accessed.
  >> Visit: https://data.statmt.org/pmindia/
PMIndia ta-en: 0 pairs saved
  PMIndia download failed: Dataset 'pmindia' doesn't exist on the Hub or cannot be accessed.
  >> Visit: https://data.statmt.org/pmindia/
PMIndia gu-en: 0 pairs saved


### 2.6 OPUS Datasets

In [10]:
def download_opus(lang_pair: str, corpora=None) -> pd.DataFrame:
    """Download OPUS parallel corpora for a language pair."""
    if corpora is None:
        corpora = ['Tatoeba', 'WikiMatrix']
    
    src_lang = lang_pair.split('-')[0]
    tgt_lang = lang_pair.split('-')[1]
    all_pairs = []
    
    for corpus_name in corpora:
        print(f'Downloading OPUS {corpus_name} for {lang_pair}...')
        try:
            ds = load_dataset(
                'opus100',
                f'en-{src_lang}' if src_lang in ('hi', 'mr', 'ta', 'gu', 'bn', 'kn', 'ml', 'or', 'pa', 'te', 'ur', 'ne', 'si', 'as') else f'{src_lang}-{tgt_lang}',
                split='train',
            )
            for item in tqdm(ds, desc=f'OPUS {corpus_name} {lang_pair}'):
                translation = item.get('translation', item)
                src = translation.get(src_lang, '')
                tgt = translation.get(tgt_lang, '')
                all_pairs.append({'source': src, 'target': tgt})
            break  # opus100 already aggregates corpora
        except Exception as e:
            print(f'OPUS {corpus_name} for {lang_pair} failed: {e}')
            try:
                ds = load_dataset(
                    f'Helsinki-NLP/opus-{corpus_name.lower()}',
                    lang1=src_lang, lang2=tgt_lang,
                    split='train'
                )
                for item in tqdm(ds, desc=f'OPUS {corpus_name}'):
                    translation = item.get('translation', item)
                    src = translation.get(src_lang, '')
                    tgt = translation.get(tgt_lang, '')
                    all_pairs.append({'source': src, 'target': tgt})
            except Exception as e2:
                print(f'  Fallback also failed: {e2}')
                continue
    
    df = pd.DataFrame(all_pairs) if all_pairs else pd.DataFrame(columns=['source', 'target'])
    save_path = DATA_RAW / lang_pair / 'opus'
    save_path.mkdir(parents=True, exist_ok=True)
    df.to_csv(save_path / 'data.tsv', sep='\t', index=False)
    
    print(f'OPUS {lang_pair}: {len(df):,} pairs saved')
    return df

opus_data = {}
for lp in CONFIG['language_pairs']:
    opus_data[lp] = download_opus(lp)

OPUS Tatoeba hi-en: 100%|██████████| 534319/534319 [00:06<00:00, 81608.58it/s]


OPUS hi-en: 534,319 pairs saved


OPUS Tatoeba mr-en: 100%|██████████| 27007/27007 [00:00<00:00, 80176.13it/s]


OPUS mr-en: 27,007 pairs saved


OPUS Tatoeba ta-en: 100%|██████████| 227014/227014 [00:02<00:00, 78807.01it/s]


OPUS ta-en: 227,014 pairs saved


OPUS Tatoeba gu-en: 100%|██████████| 318306/318306 [00:03<00:00, 80253.04it/s]


OPUS gu-en: 318,306 pairs saved


### 2.7 Custom Domain Data (Optional)

If you have custom domain-specific translation pairs (menus, signs, medical terms, etc.), place them as TSV files in `data/raw/{lang_pair}/custom/data.tsv` with columns `source` and `target`.

In [11]:
def load_custom_data(lang_pair: str) -> pd.DataFrame:
    """Load custom domain data if available."""
    custom_path = DATA_RAW / lang_pair / 'custom' / 'data.tsv'
    if custom_path.exists():
        df = pd.read_csv(custom_path, sep='\t')
        print(f'Custom data {lang_pair}: {len(df):,} pairs loaded')
        return df
    else:
        print(f'No custom data for {lang_pair} (place TSV at {custom_path})')
        return pd.DataFrame(columns=['source', 'target'])

custom_data = {}
for lp in CONFIG['language_pairs']:
    (DATA_RAW / lp / 'custom').mkdir(parents=True, exist_ok=True)
    custom_data[lp] = load_custom_data(lp)

No custom data for hi-en (place TSV at bhashalens_ml\data\raw\hi-en\custom\data.tsv)
No custom data for mr-en (place TSV at bhashalens_ml\data\raw\mr-en\custom\data.tsv)
No custom data for ta-en (place TSV at bhashalens_ml\data\raw\ta-en\custom\data.tsv)
No custom data for gu-en (place TSV at bhashalens_ml\data\raw\gu-en\custom\data.tsv)


### 2.8 Dataset Summary

In [12]:
summary_rows = []
for lp in CONFIG['language_pairs']:
    sources = {
        'samanantar': len(samanantar_data.get(lp, [])),
        'indictrans2': len(indictrans2_data.get(lp, [])),
        'indiccorp': len(indiccorp_data.get(lp, [])),
        'pmindia': len(pmindia_data.get(lp, [])),
        'opus': len(opus_data.get(lp, [])),
        'custom': len(custom_data.get(lp, [])),
    }
    if lp == 'hi-en':
        sources['iit_bombay'] = len(iit_bombay_df)
    summary_rows.append({'lang_pair': lp, **sources, 'total': sum(sources.values())})

summary_df = pd.DataFrame(summary_rows)
print('\n📊 Dataset Collection Summary:')
print(summary_df.to_string(index=False))


📊 Dataset Collection Summary:
lang_pair  samanantar  indictrans2  indiccorp  pmindia   opus  custom  iit_bombay    total
    hi-en    10125706            0          0        0 534319       0   1659083.0 12319108
    mr-en     3627480            0          0        0  27007       0         NaN  3654487
    ta-en     5264867            0          0        0 227014       0         NaN  5491881
    gu-en     3067790            0          0        0 318306       0         NaN  3386096


## 3. Data Cleaning Pipeline

Applies all 10 cleaning rules from the spec:
1. Remove empty pairs
2. Remove identical source-target pairs
3. Remove pairs exceeding 512 characters
4. Remove pairs with length ratio > 3:1
5. Remove duplicate source texts
6. Normalize Unicode to NFC
7. Remove pairs containing URLs or emails
8. Remove pairs where source is wrong language
9. Remove pairs where target is wrong language
10. Generate cleaning report

In [13]:
URL_REGEX = re.compile(
    r'https?://[\w.-]+(?:\.[\w.-]+)+[\w.,@?^=%&:/~+#-]*'
)
EMAIL_REGEX = re.compile(
    r'[\w.+-]+@[\w-]+\.[\w.-]+'
)

def detect_language_safe(text: str) -> str:
    """Detect language with fallback."""
    try:
        if len(text.strip()) < 10:
            return 'unknown'
        return detect(text)
    except Exception:
        return 'unknown'


class DataCleaner:
    """Applies all 10 cleaning rules from the BhashaLens spec."""
    
    def __init__(self, config: dict):
        self.max_length = config['max_length']
        self.max_ratio = config['max_length_ratio']
        self.stats = defaultdict(int)
    
    def clean(self, df: pd.DataFrame, src_lang: str, tgt_lang: str) -> pd.DataFrame:
        """Run full cleaning pipeline."""
        self.stats = defaultdict(int)
        initial_count = len(df)
        self.stats['initial'] = initial_count
        
        df = df.copy()
        df = df.dropna(subset=['source', 'target'])\n
        df['source'] = df['source'].astype(str)\n
        df['target'] = df['target'].astype(str)\n
        
        # Rule 1: Remove empty pairs
        mask = (df['source'].str.strip().str.len() > 0) & (df['target'].str.strip().str.len() > 0)
        removed = (~mask).sum()
        df = df[mask]
        self.stats['empty_removed'] = removed
        print(f'  Rule 1 (empty): removed {removed:,}')
        
        # Rule 2: Remove identical pairs
        mask = df['source'] != df['target']
        removed = (~mask).sum()
        df = df[mask]
        self.stats['identical_removed'] = removed
        print(f'  Rule 2 (identical): removed {removed:,}')
        
        # Rule 3: Remove pairs exceeding max length
        mask = (df['source'].str.len() <= self.max_length) & (df['target'].str.len() <= self.max_length)
        removed = (~mask).sum()
        df = df[mask]
        self.stats['too_long_removed'] = removed
        print(f'  Rule 3 (too long): removed {removed:,}')
        
        # Rule 4: Remove pairs with bad length ratio
        src_len = df['source'].str.len().clip(lower=1)
        tgt_len = df['target'].str.len().clip(lower=1)
        ratio = np.maximum(src_len, tgt_len) / np.minimum(src_len, tgt_len)
        mask = ratio <= self.max_ratio
        removed = (~mask).sum()
        df = df[mask]
        self.stats['bad_ratio_removed'] = removed
        print(f'  Rule 4 (ratio > {self.max_ratio}): removed {removed:,}')
        
        # Rule 5: Deduplicate on source text
        before = len(df)
        df = df.drop_duplicates(subset='source', keep='first')
        removed = before - len(df)
        self.stats['duplicates_removed'] = removed
        print(f'  Rule 5 (dedup): removed {removed:,}')
        
        # Rule 6: Unicode NFC normalization
        df['source'] = df['source'].apply(lambda x: unicodedata.normalize('NFC', x))
        df['target'] = df['target'].apply(lambda x: unicodedata.normalize('NFC', x))
        print(f'  Rule 6 (Unicode NFC): applied')
        
        # Rule 7: Remove pairs with URLs or emails
        has_url = df['source'].str.contains(URL_REGEX) | df['target'].str.contains(URL_REGEX)
        has_email = df['source'].str.contains(EMAIL_REGEX) | df['target'].str.contains(EMAIL_REGEX)
        mask = ~(has_url | has_email)
        removed = (~mask).sum()
        df = df[mask]
        self.stats['url_email_removed'] = removed
        print(f'  Rule 7 (URLs/emails): removed {removed:,}')
        
        # Rules 8 & 9: Language detection (sampled for speed)
        print(f'  Rules 8-9 (language detection): checking sample...')
        sample_size = min(5000, len(df))
        sample_idx = df.sample(n=sample_size, random_state=42).index
        
        bad_lang_indices = set()
        for idx in tqdm(sample_idx, desc='Language detection'):
            src_detected = detect_language_safe(df.loc[idx, 'source'])
            tgt_detected = detect_language_safe(df.loc[idx, 'target'])
            
            src_ok = src_detected in [src_lang, 'unknown']
            tgt_ok = tgt_detected in [tgt_lang, 'unknown']
            
            if not src_ok or not tgt_ok:
                bad_lang_indices.add(idx)
        
        if bad_lang_indices:
            bad_ratio = len(bad_lang_indices) / sample_size
            df = df.drop(index=list(bad_lang_indices), errors='ignore')
            self.stats['wrong_language_removed'] = len(bad_lang_indices)
            print(f'  Rules 8-9: removed {len(bad_lang_indices):,} (est. {bad_ratio:.1%} bad language)')
        else:
            self.stats['wrong_language_removed'] = 0
            print(f'  Rules 8-9: all samples passed language detection')
        
        self.stats['final'] = len(df)
        self.stats['total_removed'] = initial_count - len(df)
        self.stats['retention_rate'] = len(df) / initial_count if initial_count > 0 else 0
        
        return df.reset_index(drop=True)
    
    def get_report(self) -> dict:
        return dict(self.stats)

print('DataCleaner ready.')

DataCleaner ready.


## 4. Dataset Mixing

Mix datasets according to spec ratios:
- **Hindi-English:** 20% IIT Bombay, 25% Samanantar, 15% IndicTrans2, 10% IndicCorp, 10% PMIndia, 10% OPUS, 10% Custom
- **Others:** 25% Samanantar, 20% IndicTrans2, 15% IndicCorp, 15% PMIndia, 15% OPUS, 10% Custom

In [14]:
def mix_datasets(lang_pair: str, sources: dict, ratios: dict, target_size=None) -> pd.DataFrame:
    """Mix datasets according to specified ratios using stratified sampling."""
    print(f'\nMixing datasets for {lang_pair}...')
    
    available_sources = {k: v for k, v in sources.items() if len(v) > 0}
    available_ratios = {k: v for k, v in ratios.items() if k in available_sources}
    
    if not available_sources:
        print(f'  No data available for {lang_pair}!')
        return pd.DataFrame(columns=['source', 'target'])
    
    # Renormalize ratios for available sources
    total_ratio = sum(available_ratios.values())
    norm_ratios = {k: v / total_ratio for k, v in available_ratios.items()}
    
    if target_size is None:
        max_possible = min(
            len(available_sources[k]) / norm_ratios[k]
            for k in available_sources
        )
        target_size = int(max_possible)
    
    mixed_dfs = []
    for source_name, ratio in norm_ratios.items():
        n_samples = int(target_size * ratio)
        df = available_sources[source_name]
        
        if n_samples >= len(df):
            sampled = df.copy()
        else:
            sampled = df.sample(n=n_samples, random_state=42)
        
        sampled = sampled.copy()
        sampled['source_dataset'] = source_name
        mixed_dfs.append(sampled)
        print(f'  {source_name}: {len(sampled):,} pairs ({ratio:.0%})')
    
    mixed = pd.concat(mixed_dfs, ignore_index=True)
    mixed = mixed.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f'  Total mixed: {len(mixed):,} pairs')
    return mixed

print('Mixing function ready.')

Mixing function ready.


## 5. Train / Validation / Test Split

In [15]:
def hash_split(text: str) -> float:
    """Deterministic hash-based split value [0, 1)."""
    h = hashlib.md5(text.encode('utf-8')).hexdigest()
    return int(h[:8], 16) / 0xFFFFFFFF


def create_splits(df: pd.DataFrame, ratios=(0.90, 0.05, 0.05)) -> dict:
    """Create train/val/test splits using hash-based deterministic splitting."""
    train_end = ratios[0]
    val_end = ratios[0] + ratios[1]
    
    split_values = df['source'].apply(hash_split)
    
    train_mask = split_values < train_end
    val_mask = (split_values >= train_end) & (split_values < val_end)
    test_mask = split_values >= val_end
    
    splits = {
        'train': df[train_mask].reset_index(drop=True),
        'val': df[val_mask].reset_index(drop=True),
        'test': df[test_mask].reset_index(drop=True),
    }
    
    # Verify no overlap
    train_sources = set(splits['train']['source'])
    val_sources = set(splits['val']['source'])
    test_sources = set(splits['test']['source'])
    
    assert len(train_sources & val_sources) == 0, 'Train/val overlap!'
    assert len(train_sources & test_sources) == 0, 'Train/test overlap!'
    assert len(val_sources & test_sources) == 0, 'Val/test overlap!'
    
    print(f'  Train: {len(splits["train"]):,} | Val: {len(splits["val"]):,} | Test: {len(splits["test"]):,}')
    return splits

print('Splitting function ready.')

Splitting function ready.


## 6. Run Full Pipeline

In [16]:
cleaner = DataCleaner(CONFIG['cleaning'])
all_reports = {}

for lang_pair in CONFIG['language_pairs']:
    print(f'\n{"="*60}')
    print(f'Processing: {lang_pair}')
    print(f'{"="*60}')
    
    src_lang, tgt_lang = lang_pair.split('-')
    
    # Gather all sources for this pair
    sources = {
        'samanantar': samanantar_data.get(lang_pair, pd.DataFrame(columns=['source', 'target'])),
        'indictrans2': indictrans2_data.get(lang_pair, pd.DataFrame(columns=['source', 'target'])),
        'indiccorp': indiccorp_data.get(lang_pair, pd.DataFrame(columns=['source', 'target'])),
        'pmindia': pmindia_data.get(lang_pair, pd.DataFrame(columns=['source', 'target'])),
        'opus': opus_data.get(lang_pair, pd.DataFrame(columns=['source', 'target'])),
        'custom': custom_data.get(lang_pair, pd.DataFrame(columns=['source', 'target'])),
    }
    if lang_pair == 'hi-en':
        sources['iit_bombay'] = iit_bombay_df
    
    # Step 1: Mix
    ratios = CONFIG['mixing_ratios'][lang_pair]
    mixed_df = mix_datasets(lang_pair, sources, ratios)
    
    if len(mixed_df) == 0:
        print(f'  Skipping {lang_pair} — no data available')
        continue
    
    # Save mixed (before cleaning)
    mixed_path = DATA_MIXED / lang_pair
    mixed_path.mkdir(parents=True, exist_ok=True)
    mixed_df.to_csv(mixed_path / 'mixed.tsv', sep='\t', index=False)
    
    # Step 2: Clean
    print(f'\nCleaning {lang_pair}...')
    cleaned_df = cleaner.clean(mixed_df[['source', 'target']], src_lang, tgt_lang)
    report = cleaner.get_report()
    all_reports[lang_pair] = report
    
    # Step 3: Split
    print(f'\nSplitting {lang_pair}...')
    splits = create_splits(cleaned_df, CONFIG['split_ratios'])
    
    # Save cleaned splits
    clean_path = DATA_CLEANED / lang_pair
    clean_path.mkdir(parents=True, exist_ok=True)
    
    for split_name, split_df in splits.items():
        split_df[['source', 'target']].to_csv(
            clean_path / f'{split_name}.tsv', sep='\t', index=False
        )
    
    # Also create reverse pair (en-XX) by swapping columns
    reverse_pair = f'{tgt_lang}-{src_lang}'
    reverse_path = DATA_CLEANED / reverse_pair
    reverse_path.mkdir(parents=True, exist_ok=True)
    
    for split_name, split_df in splits.items():
        reverse_df = split_df[['target', 'source']].copy()
        reverse_df.columns = ['source', 'target']
        reverse_df.to_csv(reverse_path / f'{split_name}.tsv', sep='\t', index=False)
    
    print(f'\n✅ {lang_pair} + {reverse_pair} — done!')

print(f'\n{"="*60}')
print('All language pairs processed!')


Processing: hi-en

Mixing datasets for hi-en...
  iit_bombay: 1,068,637 pairs (36%)
  samanantar: 1,335,797 pairs (45%)
  opus: 534,318 pairs (18%)
  Total mixed: 2,938,752 pairs

Cleaning hi-en...
  Rule 1 (empty): removed 3,879
  Rule 2 (identical): removed 8,237
  Rule 3 (too long): removed 12,623
  Rule 4 (ratio > 3.0): removed 45,070
  Rule 5 (dedup): removed 848,878
  Rule 6 (Unicode NFC): applied
  Rule 7 (URLs/emails): removed 2,214
  Rules 8-9 (language detection): checking sample...


Language detection: 100%|██████████| 5000/5000 [00:16<00:00, 310.82it/s]


  Rules 8-9: removed 3,239 (est. 64.8% bad language)

Splitting hi-en...
  Train: 1,812,819 | Val: 100,939 | Test: 100,854

✅ hi-en + en-hi — done!

Processing: mr-en

Mixing datasets for mr-en...
  samanantar: 45,011 pairs (62%)
  opus: 27,006 pairs (37%)
  Total mixed: 72,017 pairs

Cleaning mr-en...
  Rule 1 (empty): removed 0
  Rule 2 (identical): removed 556
  Rule 3 (too long): removed 47
  Rule 4 (ratio > 3.0): removed 1,459
  Rule 5 (dedup): removed 3,110
  Rule 6 (Unicode NFC): applied
  Rule 7 (URLs/emails): removed 4
  Rules 8-9 (language detection): checking sample...


Language detection: 100%|██████████| 5000/5000 [00:18<00:00, 269.77it/s]


  Rules 8-9: removed 3,804 (est. 76.1% bad language)

Splitting mr-en...
  Train: 56,737 | Val: 3,141 | Test: 3,159

✅ mr-en + en-mr — done!

Processing: ta-en

Mixing datasets for ta-en...
  samanantar: 378,356 pairs (62%)
  opus: 227,013 pairs (37%)
  Total mixed: 605,369 pairs

Cleaning ta-en...
  Rule 1 (empty): removed 0
  Rule 2 (identical): removed 3,147
  Rule 3 (too long): removed 3,859
  Rule 4 (ratio > 3.0): removed 18,149
  Rule 5 (dedup): removed 179,049
  Rule 6 (Unicode NFC): applied
  Rule 7 (URLs/emails): removed 55
  Rules 8-9 (language detection): checking sample...


Language detection: 100%|██████████| 5000/5000 [00:13<00:00, 370.83it/s]


  Rules 8-9: removed 4,245 (est. 84.9% bad language)

Splitting ta-en...
  Train: 357,023 | Val: 20,107 | Test: 19,735

✅ ta-en + en-ta — done!

Processing: gu-en

Mixing datasets for gu-en...
  samanantar: 530,510 pairs (62%)
  opus: 318,305 pairs (37%)
  Total mixed: 848,815 pairs

Cleaning gu-en...
  Rule 1 (empty): removed 0
  Rule 2 (identical): removed 9,877
  Rule 3 (too long): removed 361
  Rule 4 (ratio > 3.0): removed 15,287
  Rule 5 (dedup): removed 307,794
  Rule 6 (Unicode NFC): applied
  Rule 7 (URLs/emails): removed 106
  Rules 8-9 (language detection): checking sample...


Language detection: 100%|██████████| 5000/5000 [00:14<00:00, 354.38it/s]


  Rules 8-9: removed 4,378 (est. 87.6% bad language)

Splitting gu-en...
  Train: 460,305 | Val: 25,364 | Test: 25,343

✅ gu-en + en-gu — done!

All language pairs processed!


## 7. Cleaning Reports

In [21]:
# Save cleaning reports
report_path = REPORTS_DIR / 'cleaning_report.json'
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)
with open(report_path, 'w') as f:
    json.dump(all_reports, f, indent=2, cls=NpEncoder)

print('📊 Cleaning Report Summary:')
print(f'{"":-<60}')
for lp, report in all_reports.items():
    print(f'\n{lp}:')
    print(f'  Initial:          {report["initial"]:>10,}')
    print(f'  Empty removed:    {report["empty_removed"]:>10,}')
    print(f'  Identical removed:{report["identical_removed"]:>10,}')
    print(f'  Too long removed: {report["too_long_removed"]:>10,}')
    print(f'  Bad ratio removed:{report["bad_ratio_removed"]:>10,}')
    print(f'  Duplicates:       {report["duplicates_removed"]:>10,}')
    print(f'  URL/Email:        {report["url_email_removed"]:>10,}')
    print(f'  Wrong language:   {report["wrong_language_removed"]:>10,}')
    print(f'  Final:            {report["final"]:>10,}')
    print(f'  Retention rate:   {report["retention_rate"]:>10.1%}')

📊 Cleaning Report Summary:
------------------------------------------------------------

hi-en:
  Initial:           2,938,752
  Empty removed:         3,879
  Identical removed:     8,237
  Too long removed:     12,623
  Bad ratio removed:    45,070
  Duplicates:          848,878
  URL/Email:             2,214
  Wrong language:        3,239
  Final:             2,014,612
  Retention rate:        68.6%

mr-en:
  Initial:              72,017
  Empty removed:             0
  Identical removed:       556
  Too long removed:         47
  Bad ratio removed:     1,459
  Duplicates:            3,110
  URL/Email:                 4
  Wrong language:        3,804
  Final:                63,037
  Retention rate:        87.5%

ta-en:
  Initial:             605,369
  Empty removed:             0
  Identical removed:     3,147
  Too long removed:      3,859
  Bad ratio removed:    18,149
  Duplicates:          179,049
  URL/Email:                55
  Wrong language:        4,245
  Final:            

## 8. Verification

In [25]:
print('🔍 Verifying outputs...\n')
all_ok = True

for lang_pair in CONFIG['language_pairs']:
    src_lang, tgt_lang = lang_pair.split('-')
    
    for direction in [lang_pair, f'{tgt_lang}-{src_lang}']:
        clean_path = DATA_CLEANED / direction
        
        for split in ['train', 'val', 'test']:
            fpath = clean_path / f'{split}.tsv'
            assert fpath.exists(), f'Missing: {fpath}'
            
            df = pd.read_csv(fpath, sep='\t', keep_default_na=False)
            assert len(df) > 0, f'Empty file: {fpath}'
            assert 'source' in df.columns, f'Missing source column: {fpath}'
            assert 'target' in df.columns, f'Missing target column: {fpath}'
            
            # Check no empty values
            empty_src = df['source'].isna().sum() + (df['source'].astype(str).str.strip() == '').sum()
            empty_tgt = df['target'].isna().sum() + (df['target'].astype(str).str.strip() == '').sum()
            assert empty_src == 0, f'Empty sources in {fpath}'
            assert empty_tgt == 0, f'Empty targets in {fpath}'
        
        # Check no overlap between splits
        train_df = pd.read_csv(clean_path / 'train.tsv', sep='\t', keep_default_na=False)
        val_df = pd.read_csv(clean_path / 'val.tsv', sep='\t', keep_default_na=False)
        test_df = pd.read_csv(clean_path / 'test.tsv', sep='\t', keep_default_na=False)
        
        train_src = set(zip(train_df['source'], train_df['target']))
        val_src = set(zip(val_df['source'], val_df['target']))
        test_src = set(zip(test_df['source'], test_df['target']))
        
        assert len(train_src & val_src) == 0, f'Train/val pair overlap in {direction}'
        assert len(train_src & test_src) == 0, f'Train/test pair overlap in {direction}'
        
        total = len(train_df) + len(val_df) + len(test_df)
        print(f'  ✅ {direction}: train={len(train_df):,} val={len(val_df):,} test={len(test_df):,} total={total:,}')

print(f'\n✅ All verifications passed!')
print(f'\nOutput directory: {DATA_CLEANED}')
print('These files will be used by Notebook 2 (Training).')

🔍 Verifying outputs...

  ✅ hi-en: train=1,812,819 val=100,939 test=100,854 total=2,014,612
  ✅ en-hi: train=1,812,819 val=100,939 test=100,854 total=2,014,612
  ✅ mr-en: train=56,737 val=3,141 test=3,159 total=63,037
  ✅ en-mr: train=56,737 val=3,141 test=3,159 total=63,037
  ✅ ta-en: train=357,023 val=20,107 test=19,735 total=396,865
  ✅ en-ta: train=357,023 val=20,107 test=19,735 total=396,865
  ✅ gu-en: train=460,305 val=25,364 test=25,343 total=511,012
  ✅ en-gu: train=460,305 val=25,364 test=25,343 total=511,012

✅ All verifications passed!

Output directory: bhashalens_ml\data\cleaned
These files will be used by Notebook 2 (Training).
